In [5]:
import pandas as pd
from pathlib import Path

def scan_aois(root_folder):
    """
    Scan all CSV/TSV/TXT files under root_folder for AOI values.
    Returns a DataFrame with one row per file.
    """
    root = Path(root_folder)
    files = sorted([p for p in root.rglob("*") if p.suffix.lower() in [".csv", ".tsv", ".txt"]])
    
    results = []
    for f in files:
        try:
            # auto-detect delimiter
            for sep in [",", "\t", ";", "|"]:
                try:
                    df = pd.read_csv(f, sep=sep, engine="python")
                    break
                except Exception:
                    df = None
            if df is None:
                raise ValueError("Could not read with any common delimiter")

            if "AOI" not in df.columns:
                results.append({"file": str(f), "has_aoi": False, "aois": []})
                continue

            # clean AOI column
            aois = df["AOI"].dropna().astype(str).str.strip()
            aois = aois[~aois.str.lower().isin(["", "nan", "none", "null"])]
            unique_aois = sorted(aois.unique().tolist())

            results.append({
                "file": str(f),
                "has_aoi": len(unique_aois) > 0,
                "aois": unique_aois
            })
        except Exception as e:
            results.append({"file": str(f), "has_aoi": False, "aois": [], "error": str(e)})
    
    return pd.DataFrame(results)

# 👉 Run the scan
summary = scan_aois("/Users/ferolahima/Desktop/gazepoint_project/data")

# Split into two groups
with_aois = summary[summary["has_aoi"] == True]
without_aois = summary[summary["has_aoi"] == False]

print("=== Files WITH AOIs ===")
for i in range(len(with_aois)):
    print(f"{with_aois.iloc[i]['file']}: {with_aois.iloc[i]['aois']}")
print("\n=== Files WITHOUT AOIs ===")
for i in range(len(without_aois)):
    print(f"{without_aois.iloc[i]['file']}")        
# display(with_aois)

# print("\n=== Files WITHOUT AOIs ===")
# display(without_aois)

=== Files WITH AOIs ===
/Users/ferolahima/Desktop/gazepoint_project/data/P11_S2_ITA/all_gaze/User 0_all_gaze.csv: ['P11_ALL', 'P11_LIST-P11_ALL', 'P11_MAP', 'P11_MAP-P11_ALL']
/Users/ferolahima/Desktop/gazepoint_project/data/P11_S2_ITA/fixations/User 0_fixations.csv: ['P11_ALL', 'P11_LIST-P11_ALL', 'P11_MAP-P11_ALL']
/Users/ferolahima/Desktop/gazepoint_project/data/P32_S1_Ita/all_gaze/User 0_all_gaze.csv: ['P32_ALL', 'P32_LIST', 'P32_LIST-P32_ALL', 'P32_MAP', 'P32_MAP-P32_ALL']
/Users/ferolahima/Desktop/gazepoint_project/data/P32_S1_Ita/fixations/User 0_fixations.csv: ['P32_ALL', 'P32_LIST-P32_ALL', 'P32_MAP', 'P32_MAP-P32_ALL']
/Users/ferolahima/Desktop/gazepoint_project/data/P3_S2_ENG/all_gaze/User 0_all_gaze.csv: ['P3_ALL', 'P3_LIST-P3_ALL', 'P3_MAP', 'P3_MAP-P3_ALL']
/Users/ferolahima/Desktop/gazepoint_project/data/P3_S2_ENG/fixations/User 0_fixations.csv: ['P3_ALL', 'P3_LIST-P3_ALL', 'P3_MAP', 'P3_MAP-P3_ALL']
/Users/ferolahima/Desktop/gazepoint_project/data/P8_S1_ITA/all_gaze/Use